# Trust Game Training (HF Space Remote)

This notebook trains with **TRL + Unsloth** while connecting to your deployed environment on Hugging Face Space (no local server imports).

- Space: [hardikshreyas/trust_game_env](https://huggingface.co/spaces/hardikshreyas/trust_game_env)
- Recommended runtime: Colab GPU (T4+)

In [1]:
# Install dependencies
%pip install -q --retries 10 --timeout 120 unsloth trl transformers datasets accelerate peft bitsandbytes matplotlib pandas requests

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import random
from typing import Dict, Any, List

import requests
import matplotlib.pyplot as plt
from datasets import Dataset

from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)

BASE_URL = "https://hardikshreyas-trust-game-env.hf.space"
print("Using env:", BASE_URL)

In [ ]:
def reset_env() -> Dict[str, Any]:
    r = requests.post(f"{BASE_URL}/reset", timeout=60)
    r.raise_for_status()
    return r.json()

def safe_action_from_obs(obs: Dict[str, Any]) -> Dict[str, Any]:
    # Minimal action to avoid fragile env branches during remote data collection.
    return {
        "agent_id": int(obs.get("your_agent_id", 0)),
        "claim_amount": float(max(0.0, min(100.0, obs.get("your_true_need", 40.0)))),
        "verify_targets": [],
        "accept_proposal": False,
        "message": "safe-fallback",
    }

def step_env(action: Dict[str, Any], retries: int = 2) -> Dict[str, Any] | None:
    # OpenEnv /step expects StepRequest shape: {"action": <action_dict>}
    payload = {"action": action}
    for attempt in range(retries + 1):
        r = requests.post(f"{BASE_URL}/step", json=payload, timeout=60)
        if r.status_code < 400:
            return r.json()
        print(f"Step error status: {r.status_code} (attempt {attempt+1}/{retries+1})")
        print("Step error body:", r.text[:800])
    return None

def heuristic_action(obs: Dict[str, Any]) -> Dict[str, Any]:
    role = obs.get("your_role", "honest")
    agent_id = int(obs.get("your_agent_id", 0))
    need = float(obs.get("your_true_need", 40.0))
    trust_scores = obs.get("trust_scores", {}) or {}

    others = [(int(k), float(v)) for k, v in trust_scores.items() if int(k) != agent_id]
    others_sorted = [k for k, _ in sorted(others, key=lambda x: x[1])]

    claim = need
    verify_targets: List[int] = []
    accept = False

    if role == "honest":
        claim = need
        verify_targets = others_sorted[:2]
        accept = float(obs.get("system_metrics", {}).get("fairness", 0.0)) >= 0.7
    elif role == "self_interested":
        claim = min(100.0, need + 10.0)
        accept = True
    elif role == "adversarial":
        claim = min(100.0, need + 25.0)
        accept = True
    elif role == "victim":
        claim = need
        verify_targets = [k for k, v in others if v < 0.4]
        accept = True
    elif role == "oversight":
        claim = need
        verify_targets = others_sorted[:2]
        accept = float(obs.get("system_metrics", {}).get("deception_rate", 1.0)) < 0.1

    return {
        "agent_id": agent_id,
        "claim_amount": float(max(0.0, min(100.0, claim))),
        "verify_targets": verify_targets,
        "accept_proposal": bool(accept),
        "message": "auto-policy",
    }

# quick connectivity check
sample = reset_env()
print("Reset keys:", sample.keys())
print("Observation keys:", sample.get("observation", {}).keys())

In [ ]:
def generate_remote_dataset(n_episodes: int = 120) -> Dataset:
    rows = []
    skipped_episodes = 0

    for _ in range(n_episodes):
        payload = reset_env()
        obs = payload.get("observation", {})
        done = bool(payload.get("done", False))

        steps = 0
        bad_episode = False
        while not done and steps < 64:
            action = heuristic_action(obs)
            rows.append({
                "prompt": obs.get("prompt", ""),
                "response": json.dumps({
                    "agent_id": action["agent_id"],
                    "claim_amount": action["claim_amount"],
                    "verify_targets": action["verify_targets"],
                    "accept_proposal": action["accept_proposal"],
                })
            })

            next_payload = step_env(action)
            if next_payload is None:
                # Retry same step with a conservative fallback action.
                next_payload = step_env(safe_action_from_obs(obs))

            if next_payload is None or "observation" not in next_payload:
                bad_episode = True
                break

            obs = next_payload["observation"]
            done = bool(next_payload.get("done", False))
            steps += 1

        if bad_episode:
            skipped_episodes += 1

    print(f"Collected {len(rows)} training rows, skipped episodes: {skipped_episodes}")
    ds = Dataset.from_list(rows)
    return ds.train_test_split(test_size=0.1, seed=SEED)

dataset = generate_remote_dataset(120)
dataset

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

def format_example(example):
    text = (
        "You are an agent in the Trust Game environment.\n"
        "Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal.\n\n"
        f"Prompt:\n{example['prompt']}\n\nAction JSON:\n{example['response']}"
    )
    return {"text": text}

train_ds = dataset["train"].map(format_example)
eval_ds = dataset["test"].map(format_example)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    args=SFTConfig(
        output_dir="outputs/trust-game-sft-remote",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        bf16=True,
        report_to="none",
    ),
)

trainer.train()

In [ ]:
def parse_action_json(text: str, fallback_agent_id: int) -> Dict[str, Any]:
    try:
        l = text.find("{")
        r = text.rfind("}")
        obj = json.loads(text[l:r+1])
        return {
            "agent_id": int(obj.get("agent_id", fallback_agent_id)),
            "claim_amount": float(obj.get("claim_amount", 40.0)),
            "verify_targets": [int(x) for x in obj.get("verify_targets", [])],
            "accept_proposal": bool(obj.get("accept_proposal", False)),
            "message": "trained-model",
        }
    except Exception:
        return {
            "agent_id": fallback_agent_id,
            "claim_amount": 40.0,
            "verify_targets": [],
            "accept_proposal": False,
            "message": "fallback",
        }

def model_action(obs: Dict[str, Any]) -> Dict[str, Any]:
    prompt = (
        "Return ONLY valid JSON with keys: agent_id, claim_amount, verify_targets, accept_proposal.\n\n"
        f"Prompt:\n{obs.get('prompt', '')}\n\nAction JSON:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=120)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return parse_action_json(text, fallback_agent_id=int(obs.get("your_agent_id", 0)))

def run_remote_eval(policy_kind="heuristic", n_episodes=20):
    rewards = []
    for _ in range(n_episodes):
        payload = reset_env()
        obs = payload["observation"]
        done = bool(payload.get("done", False))
        total_reward = 0.0
        steps = 0
        while not done and steps < 64:
            action = heuristic_action(obs) if policy_kind == "heuristic" else model_action(obs)
            payload = step_env(action)
            total_reward += float(payload.get("reward") or 0.0)
            obs = payload["observation"]
            done = bool(payload.get("done", False))
            steps += 1
        rewards.append(total_reward)
    return rewards

baseline_rewards = run_remote_eval("heuristic", 20)
trained_rewards = run_remote_eval("model", 20)

print("Baseline mean reward:", sum(baseline_rewards) / len(baseline_rewards))
print("Trained mean reward :", sum(trained_rewards) / len(trained_rewards))

plt.figure(figsize=(7, 4))
plt.plot(baseline_rewards, label="Baseline", marker="o")
plt.plot(trained_rewards, label="Trained", marker="o")
plt.xlabel("Episode")
plt.ylabel("Total reward (sum of step rewards)")
plt.title("Trust Game (HF Space): Baseline vs Trained")
plt.legend()
plt.tight_layout()

import os
os.makedirs("eval_results", exist_ok=True)
plot_path = "eval_results/training_reward_comparison_remote.png"
plt.savefig(plot_path, dpi=180)
print("Saved:", plot_path)

In [ ]:
trainer.save_model("outputs/trust-game-sft-remote/final")
tokenizer.save_pretrained("outputs/trust-game-sft-remote/final")
print("Saved model to outputs/trust-game-sft-remote/final")